In [1]:
import glob

In [2]:
from echotab.models.EchoTabInference import EchoTabInference

In [58]:
from echotab.util.echoprime import load_echoprime_clip

### Load model

In [3]:
model = EchoTabInference(
    ckpt_dir='../checkpoints',
    view_classifier_path='../model_weights/view_classifier.pt',
)

Initializing the echo branch...
The EchoPrime echo encoder is randomly initialized!
Initializing the tabular branch...
Initializing the tabular decoder...
Successfully loaded echo_branch.ckpt: <All keys matched successfully>
Successfully loaded tab_branch.ckpt: <All keys matched successfully>
Successfully loaded tab_decoder.ckpt: <All keys matched successfully>
Successfully loaded echo_projection.ckpt: <All keys matched successfully>
Successfully loaded tab_projection.ckpt: <All keys matched successfully>
Successfully loaded prediction_head.ckpt: <All keys matched successfully>


In [4]:
model

EchoTabInference(
  (encoder): EchoTabEncoder(
    (echo_branch): EchoBranch(
      (echo_encoder): EchoPrime(
        (model): MViT(
          (conv_proj): Conv3d(3, 96, kernel_size=(3, 7, 7), stride=(2, 4, 4), padding=(1, 3, 3))
          (pos_encoding): PositionalEncoding()
          (blocks): ModuleList(
            (0): MultiscaleBlock(
              (norm1): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
              (norm2): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
              (attn): MultiscaleAttention(
                (qkv): Linear(in_features=96, out_features=288, bias=True)
                (project): Sequential(
                  (0): Linear(in_features=96, out_features=96, bias=True)
                )
                (pool_q): Pool(
                  (pool): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), groups=96, bias=False)
                  (norm_act): Sequential(
                    (0): LayerNorm(

In [5]:
model.labels
# all the measurements the model can predict

['ar_pr_halftime',
 'arch_diam',
 'ascending_diam',
 'av_area_continuity',
 'av_mean_grad',
 'av_pk_grad',
 'av_pk_vel',
 'av_vti',
 'biplane_lvedv',
 'biplane_lvef',
 'biplane_lvesv',
 'descending_diam',
 'e_e_prime',
 'fs',
 'inf_lat_thickness',
 'ivc_diam',
 'la_4ch_length',
 'la_dimen',
 'la_vol',
 'lat_e_prime',
 'lateral_E_to_eprime',
 'lvedd',
 'lvesd',
 'lvot_diam',
 'lvot_pk_vel',
 'lvot_vti',
 'mean_E_to_eprime',
 'mv_e_decel',
 'mv_mean_grad',
 'mv_peak_a',
 'mv_peak_e',
 'mv_peak_e_a',
 'mv_peak_vel',
 'mva2',
 'pv_peak_vel',
 'ra_length',
 'rest_lvot',
 'rv_diam',
 'sept_e_prime',
 'septal_E_to_eprime',
 'septal_thickness',
 'sinus_diam',
 'tapse',
 'tr_mmhg',
 'tr_velocity']

### Predict measurements and extract measurement tokens

### Option 1:

Each clip is assigned to its most likely view type. Per view type, the clip with the highest confidence is used. Views that are not found are treated as missing.

In [32]:
#   meas_tokens: (1, N_measurements, 768), in the order of model.labels
#   meas_preds:  (1, N_measurements), in the units of data/mimic/phenotype_list.csv
#   assignments: per clip: path, view, confidence, selected
meas_tokens, meas_preds, assignments = model(glob.glob('../sample_study/*.avi'))

In [33]:
meas_tokens

tensor([[[-0.2061, -0.9751, -0.3729,  ..., -0.2343, -0.2369,  0.2551],
         [-0.4538, -0.7809, -1.2102,  ..., -0.9327,  0.8669,  0.2858],
         [-0.0280, -0.7107, -0.1869,  ..., -1.7186,  1.6988, -0.0770],
         ...,
         [-0.4138, -0.9805,  0.0367,  ..., -0.1668,  1.1526,  0.2842],
         [-0.2880, -0.7639, -0.7183,  ...,  0.1502,  0.3751,  0.2298],
         [ 0.3488,  0.2667, -1.3103,  ..., -0.3368, -0.1230,  0.5087]]])

In [34]:
meas_preds

tensor([[4.0932e-01, 2.5132e+00, 2.8140e+00, 2.3922e+00, 6.3281e+00, 1.1529e+01,
         1.2789e+00, 2.8402e+01, 6.9038e+01, 5.8736e+01, 2.6769e+01, 1.8475e+00,
         1.1729e+01, 3.3547e+01, 1.0404e+00, 1.3093e+00, 4.4107e+00, 3.2613e+00,
         3.7681e+01, 7.0171e-02, 1.1323e+01, 3.7527e+00, 2.4046e+00, 1.9456e+00,
         1.0704e+00, 1.8032e+01, 1.1207e+01, 1.8979e-01, 2.2232e+00, 1.0279e+00,
         6.2257e-01, 6.8147e-01, 1.1983e+00, 3.8227e+00, 9.7555e-01, 4.1436e+00,
         1.2594e+01, 2.5520e+00, 6.2848e-02, 1.3389e+01, 1.1175e+00, 2.7815e+00,
         1.7151e+00, 1.9997e+01, 2.3311e+00]])

In [35]:
assignments

,path,view,confidence,selected
0,../sample_study/p11459825_s96143613_96143613_0...,Parasternal_Short,0.999113,False
1,../sample_study/p11459825_s96143613_96143613_0...,Apical_Doppler,0.999217,False
2,../sample_study/p11459825_s96143613_96143613_0...,Apical_Doppler,0.999087,False
3,../sample_study/p11459825_s96143613_96143613_0...,Doppler_Parasternal_Long,0.997396,False
4,../sample_study/p11459825_s96143613_96143613_0...,Apical_Doppler,0.512537,False
5,../sample_study/p11459825_s96143613_96143613_0...,Parasternal_Short,0.998781,False
6,../sample_study/p11459825_s96143613_96143613_0...,A5C,0.944693,True
7,../sample_study/p11459825_s96143613_96143613_0...,Parasternal_Long,0.988351,True
8,../sample_study/p11459825_s96143613_96143613_0...,Subcostal,0.507359,False
9,../sample_study/p11459825_s96143613_96143613_0...,SSN,0.701844,False


In [36]:
assignments[assignments['selected']]

,path,view,confidence,selected
6,../sample_study/p11459825_s96143613_96143613_0...,A5C,0.944693,True
7,../sample_study/p11459825_s96143613_96143613_0...,Parasternal_Long,0.988351,True
13,../sample_study/p11459825_s96143613_96143613_0...,A2C,0.905649,True
14,../sample_study/p11459825_s96143613_96143613_0...,A4C,0.992625,True
17,../sample_study/p11459825_s96143613_96143613_0...,A3C,0.990426,True
18,../sample_study/p11459825_s96143613_96143613_0...,Apical_Doppler,0.999739,True
23,../sample_study/p11459825_s96143613_96143613_0...,Parasternal_Short,0.999307,True


In [56]:
assignments[assignments['selected']].loc[13]['path']

'../sample_study/p11459825_s96143613_96143613_0019.avi'

In [37]:
lvef_idx = model.labels.index('biplane_lvef')
lvef_token = meas_tokens[:, lvef_idx]  # (1, 768)
lvef = meas_preds[0, lvef_idx].item()  # LVEF in %
lvef

58.736331939697266

In [39]:
# All predictions
pd.DataFrame({'measurement': model.labels, 'pred': meas_preds.squeeze().tolist()})

,measurement,pred
0,ar_pr_halftime,0.409321
1,arch_diam,2.513156
2,ascending_diam,2.814018
3,av_area_continuity,2.392192
4,av_mean_grad,6.328115
5,av_pk_grad,11.529264
6,av_pk_vel,1.278865
7,av_vti,28.402376
8,biplane_lvedv,69.037605
9,biplane_lvef,58.736332


### Option 2:

In [59]:
meas_tokens, meas_preds = model.predict({
    'A2C': load_echoprime_clip('../sample_study/p11459825_s96143613_96143613_0019.avi').unsqueeze(0),
    'A4C': load_echoprime_clip('../sample_study/p11459825_s96143613_96143613_0018.avi').unsqueeze(0),
})

In [60]:
meas_tokens

tensor([[[-0.3617, -1.2213, -0.2727,  ..., -0.2663, -0.3360,  0.0878],
         [-0.6174, -0.9811, -1.1273,  ..., -0.9543,  0.9145, -0.0839],
         [-0.0498, -0.9795,  0.0308,  ..., -1.5850,  1.5464, -0.5402],
         ...,
         [-0.6880, -1.1798, -0.0031,  ..., -0.2598,  0.7659,  0.2153],
         [-0.5256, -0.8193, -0.7300,  ...,  0.0804,  0.3320,  0.1887],
         [-0.2525, -0.0149, -1.2874,  ..., -0.2785, -0.3062,  0.4234]]])

In [61]:
meas_preds

tensor([[3.4478e-01, 2.5444e+00, 2.8401e+00, 2.0267e+00, 8.2162e+00, 1.5266e+01,
         1.3580e+00, 2.9687e+01, 7.7041e+01, 5.3288e+01, 3.2951e+01, 1.7534e+00,
         1.1452e+01, 2.8749e+01, 1.0236e+00, 1.2216e+00, 4.1574e+00, 3.4230e+00,
         3.5895e+01, 5.6470e-02, 1.0563e+01, 3.6651e+00, 2.4901e+00, 1.9406e+00,
         9.8558e-01, 1.5783e+01, 1.0893e+01, 1.8321e-01, 1.9444e+00, 1.0276e+00,
         5.1570e-01, 5.5479e-01, 1.0491e+00, 3.3143e+00, 9.5379e-01, 3.9006e+00,
         1.1260e+01, 2.4847e+00, 5.3477e-02, 1.3630e+01, 1.1585e+00, 2.7808e+00,
         1.4904e+00, 1.7924e+01, 2.1469e+00]])